In [ ]:
import xml.etree.ElementTree as ET
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
import numpy as np
import joblib
import os

ModuleNotFoundError: No module named 'transformers'

In [ ]:
tree4 = ET.parse("../datasets/en_product4.xml")
root4 = tree4.getroot()

data = []

for disorder in root4.iter("Disorder"):
    name_el = disorder.find("Name")
    if name_el is None:
        continue
    disease_name = name_el.text.strip()
    
    symptoms = []
    for hpo in disorder.iter("HPOTerm"):
        if hpo.text:
            symptoms.append(hpo.text.strip())
    
    if len(symptoms) >= 3:
        text = f"{disease_name}: " + ", ".join(symptoms)
        data.append({"text": text, "label": disease_name})

print(f"Total samples: {len(data)}")

In [ ]:
le = LabelEncoder()
texts = [d["text"] for d in data]
labels = le.fit_transform([d["label"] for d in data])

print(f"Number of classes: {len(le.classes_)}")

joblib.dump(le, "../models_saved/biobert_label_encoder.pkl")

In [ ]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

print("Tokenization complete")
print(f"Input IDs shape: {encodings['input_ids'].shape}")

In [ ]:
from torch.utils.data import Dataset

class DiseaseDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

dataset = DiseaseDataset(encodings, labels)
print(f"Dataset size: {len(dataset)}")


In [ ]:
num_labels = len(le.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

print(f"Model loaded with {num_labels} output labels")

In [ ]:
training_args = TrainingArguments(
    output_dir="../models_saved/biobert_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=1,
    logging_steps=100,
    learning_rate=2e-5,
    warmup_steps=100,
    no_cuda=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete")

In [ ]:
model.save_pretrained("../models_saved/biobert_finetuned")
tokenizer.save_pretrained("../models_saved/biobert_finetuned")
print("BioBERT saved to models_saved/biobert_finetuned/")

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="../models_saved/biobert_finetuned",
    tokenizer="../models_saved/biobert_finetuned",
    device=-1
)

test_text = "Wilson Disease: tremor, liver dysfunction, Kayser-Fleischer rings, speech difficulty"
result = classifier(test_text, top_k=3)
print("Test prediction:")
for r in result:
    label_idx = int(r["label"].split("_")[-1])
    print(f"  {le.classes_[label_idx]} — {round(r['score']*100, 2)}%")